In [1]:
print("Hi")

Hi


In [2]:
import sys

print("Notebook Python:", sys.version)
if sys.version_info >= (3, 14):
    print("WARNING: Python 3.14 may not support faiss-cpu/sentence-transformers yet. Select a Python 3.11 or 3.12 kernel if install fails.")

!{sys.executable} -m pip install -q faiss-cpu groq langchain-core langchain-community langchain-text-splitters langchain-groq langchain-huggingface sentence-transformers pypdf requests beautifulsoup4


Notebook Python: 3.14.3 (v3.14.3:323c59a5e34, Feb  3 2026, 11:41:37) [Clang 16.0.0 (clang-1600.0.26.6)]

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip3.14 install --upgrade pip


In [3]:
import os
import re
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/k8/kc8jrng519q81_pvhp9qvnyr0000gn/T/ipykernel_75542/3556572008.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
GROQ_API_KEY = "yourapikey"

if GROQ_API_KEY != "your_groq_api_key_here":
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

if not os.getenv("GROQ_API_KEY"):
    print("Add your Groq API key before running answer-generation cells.")
else:
    print("Groq API key loaded.")


Groq API key loaded.


In [5]:
PDF_PATH = Path(r"/Users/santhoshkumarv/vs_code_projects/internship-harshith/projects/capstone_project/data/fia_2026_formula_1_technical_regulations_issue_8_-_2024-06-24-5.pdf")

WEB_URLS = [
    "https://en.wikipedia.org/wiki/2023_Monaco_Grand_Prix",
    "https://en.wikipedia.org/wiki/Max_Verstappen",
    "https://en.wikipedia.org/wiki/Lewis_Hamilton",
    "https://en.wikipedia.org/wiki/Michael_Schumacher",
    "https://en.wikipedia.org/wiki/Pit_stop",
    "https://en.wikipedia.org/wiki/Formula_One",
    "https://en.wikipedia.org/wiki/Formula_One_car",
]

print("PDF exists:", PDF_PATH.exists())
print("Website sources:", len(WEB_URLS))


PDF exists: True
Website sources: 7


In [6]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def load_website_document(url):
    headers = {
        "User-Agent": "Mozilla/5.0 F1 Capstone RAG Notebook"
    }
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "header", "sup"]):
        tag.decompose()

    title = soup.title.get_text(" ", strip=True) if soup.title else url
    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all(["p", "li", "h1", "h2", "h3"])]
    text = clean_text(" ".join(paragraphs))

    return Document(
        page_content=text,
        metadata={"source": url, "title": title, "type": "website"},
    )


def load_pdf_documents(pdf_path, max_pages=None):
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    reader = PdfReader(str(pdf_path))
    documents = []

    total_pages = len(reader.pages)
    pages_to_read = total_pages if max_pages is None else min(total_pages, max_pages)

    for page_number in range(pages_to_read):
        page = reader.pages[page_number]
        text = clean_text(page.extract_text() or "")

        if len(text) < 50:
            continue

        documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": str(pdf_path),
                    "title": "FIA 2026 Formula 1 Technical Regulations",
                    "type": "pdf",
                    "page": page_number + 1,
                },
            )
        )

    return documents


In [7]:
documents = []

for url in WEB_URLS:
    try:
        doc = load_website_document(url)
        if len(doc.page_content) > 200:
            documents.append(doc)
            print("Loaded website:", url)
        else:
            print("Skipped short website:", url)
    except Exception as error:
        print("Website failed:", url)
        print("Reason:", error)

try:
    pdf_docs = load_pdf_documents(PDF_PATH)
    documents.extend(pdf_docs)
    print(f"Loaded PDF pages: {len(pdf_docs)}")
except Exception as error:
    print("PDF loading failed:", error)

print("Total source documents:", len(documents))


Loaded website: https://en.wikipedia.org/wiki/2023_Monaco_Grand_Prix
Loaded website: https://en.wikipedia.org/wiki/Max_Verstappen
Loaded website: https://en.wikipedia.org/wiki/Lewis_Hamilton
Loaded website: https://en.wikipedia.org/wiki/Michael_Schumacher
Loaded website: https://en.wikipedia.org/wiki/Pit_stop
Loaded website: https://en.wikipedia.org/wiki/Formula_One
Loaded website: https://en.wikipedia.org/wiki/Formula_One_car
Loaded PDF pages: 207
Total source documents: 214


In [8]:
for index, doc in enumerate(documents[:8], start=1):
    print(f"{index}. {doc.metadata.get('type')} | {doc.metadata.get('title')} | {doc.metadata.get('source')}")
    if doc.metadata.get("page"):
        print("   Page:", doc.metadata.get("page"))


1. website | 2023 Monaco Grand Prix - Wikipedia | https://en.wikipedia.org/wiki/2023_Monaco_Grand_Prix
2. website | Max Verstappen - Wikipedia | https://en.wikipedia.org/wiki/Max_Verstappen
3. website | Lewis Hamilton - Wikipedia | https://en.wikipedia.org/wiki/Lewis_Hamilton
4. website | Michael Schumacher - Wikipedia | https://en.wikipedia.org/wiki/Michael_Schumacher
5. website | Pit stop - Wikipedia | https://en.wikipedia.org/wiki/Pit_stop
6. website | Formula One - Wikipedia | https://en.wikipedia.org/wiki/Formula_One
7. website | Formula One car - Wikipedia | https://en.wikipedia.org/wiki/Formula_One_car
8. pdf | FIA 2026 Formula 1 Technical Regulations | /Users/santhoshkumarv/vs_code_projects/internship-harshith/projects/capstone_project/data/fia_2026_formula_1_technical_regulations_issue_8_-_2024-06-24-5.pdf
   Page: 1


In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=180,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))


Total chunks: 1732


In [10]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embedding_model)
retriever = vector_db.as_retriever(search_kwargs={"k": 6})

print("FAISS vector database is ready.")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11010.36it/s]


FAISS vector database is ready.


In [11]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
)

prompt = PromptTemplate.from_template(
    """
You are a Formula 1 expert assistant for a capstone project.
Use the retrieved context to answer the user's question.

Answer style:
- Start with a direct 1-2 sentence answer.
- Then give neat bullet points with the main reasons or comparison points.
- If the question asks for a comparison, use a compact table when useful.
- Mention whether the answer comes from website data, FIA PDF data, or both.
- Do not invent facts that are not supported by the context.

Context:
{context}

Question:
{question}

Neat answer:
"""
)

print("Groq LLM connected.")


Groq LLM connected.


In [13]:
def format_source(doc):
    source_type = doc.metadata.get("type", "source")
    title = doc.metadata.get("title", "Untitled")
    source = doc.metadata.get("source", "")
    page = doc.metadata.get("page")

    if page:
        return f"{source_type.upper()} | {title} | page {page}"
    return f"{source_type.upper()} | {title} | {source}"



In [14]:

def format_docs(docs):
    formatted = []
    for doc in docs:
        source = format_source(doc)
        formatted.append(f"Source: {source}\nContent: {doc.page_content}")
    return "\n\n---\n\n".join(formatted)



In [15]:

def ask_f1_chatbot(question, show_sources=True):
    retrieved_docs = retriever.invoke(question)
    context = format_docs(retrieved_docs)
    final_prompt = prompt.format(context=context, question=question)
    response = llm.invoke(final_prompt)

    print("Question:", question)
    print("\nAnswer:\n")
    print(response.content)

    if show_sources:
        print("\nSources used:")
        seen = set()
        for doc in retrieved_docs:
            source_text = format_source(doc)
            if source_text not in seen:
                print("-", source_text)
                seen.add(source_text)

    return {
        "question": question,
        "answer": response.content,
        "source_documents": retrieved_docs,
    }

print("RAG chatbot is ready.")


RAG chatbot is ready.


In [16]:
ask_f1_chatbot("Why did Max Verstappen win Monaco 2023?")


Question: Why did Max Verstappen win Monaco 2023?

Answer:

Max Verstappen won the 2023 Monaco Grand Prix due to his pole position and his ability to maintain his lead despite the rain-affected conditions.

Here are the main reasons for his win:

* **Pole position**: Verstappen secured the fastest lap in qualifying, giving him a significant advantage at the start of the race.
* **Rain-affected conditions**: The rain during the middle stages of the race made it difficult for other drivers to overtake Verstappen, allowing him to maintain his lead.
* **Experience and skill**: Verstappen's experience and skill in wet-weather conditions helped him to navigate the challenging track and stay ahead of his competitors.

Data source: Wikipedia articles on Max Verstappen and the 2023 Monaco Grand Prix.

Sources used:
- WEBSITE | Max Verstappen - Wikipedia | https://en.wikipedia.org/wiki/Max_Verstappen
- WEBSITE | 2023 Monaco Grand Prix - Wikipedia | https://en.wikipedia.org/wiki/2023_Monaco_Grand

{'question': 'Why did Max Verstappen win Monaco 2023?',
 'answer': "Max Verstappen won the 2023 Monaco Grand Prix due to his pole position and his ability to maintain his lead despite the rain-affected conditions.\n\nHere are the main reasons for his win:\n\n* **Pole position**: Verstappen secured the fastest lap in qualifying, giving him a significant advantage at the start of the race.\n* **Rain-affected conditions**: The rain during the middle stages of the race made it difficult for other drivers to overtake Verstappen, allowing him to maintain his lead.\n* **Experience and skill**: Verstappen's experience and skill in wet-weather conditions helped him to navigate the challenging track and stay ahead of his competitors.\n\nData source: Wikipedia articles on Max Verstappen and the 2023 Monaco Grand Prix.",
 'source_documents': [Document(id='fb363e95-e4fd-4b78-adff-6702c44edfef', metadata={'source': 'https://en.wikipedia.org/wiki/Max_Verstappen', 'title': 'Max Verstappen - Wikipedia'

In [17]:
ask_f1_chatbot("What do the 2026 FIA Formula 1 technical regulations say about car design or power units?")


Question: What do the 2026 FIA Formula 1 technical regulations say about car design or power units?

Answer:

The 2026 FIA Formula 1 technical regulations emphasize the importance of car design and power unit compliance, with a focus on safety and regulation adherence. The regulations outline specific guidelines for power unit manufacturers and competitors to ensure that cars meet the required standards.

Main points:

* The stewards may prohibit a vehicle's participation if its construction is deemed to be dangerous (FIA 2026 Formula 1 Technical Regulations, page 8).
* Formula 1 cars must comply with the regulations in their entirety at all times during a competition (FIA 2026 Formula 1 Technical Regulations, page 8).
* Power units are subject to prior approval from PU manufacturers, as per the 2026 F1 PU Governance Agreement (FIA 2026 Formula 1 Technical Regulations, page 8).
* New power unit elements must include all modifications included in any power unit element already used by t

{'question': 'What do the 2026 FIA Formula 1 technical regulations say about car design or power units?',
 'answer': "The 2026 FIA Formula 1 technical regulations emphasize the importance of car design and power unit compliance, with a focus on safety and regulation adherence. The regulations outline specific guidelines for power unit manufacturers and competitors to ensure that cars meet the required standards.\n\nMain points:\n\n* The stewards may prohibit a vehicle's participation if its construction is deemed to be dangerous (FIA 2026 Formula 1 Technical Regulations, page 8).\n* Formula 1 cars must comply with the regulations in their entirety at all times during a competition (FIA 2026 Formula 1 Technical Regulations, page 8).\n* Power units are subject to prior approval from PU manufacturers, as per the 2026 F1 PU Governance Agreement (FIA 2026 Formula 1 Technical Regulations, page 8).\n* New power unit elements must include all modifications included in any power unit element al